# Colab runner: one click per experiment

Open this notebook in Colab and choose **Runtime > Run all**. Never edit a cell: the job to run is read from the `colab-queue` branch
(`job.json`), the result bundle is pushed to the `colab-results` branch and copied to Drive (`MyDrive/crypto_runs/<job_id>/`).

First run only: paste the GitHub token when asked (it is not echoed); it is checked against the repo and saved to
`MyDrive/crypto/secrets/github_token.txt`, and every later run reads it silently. The `job.json` schema and how results are read:
`docs/research/plan/COLAB_RUNNER.md`.
All logic lives in `tools/colab_runner.py`; this notebook only mounts Drive and calls it. The next code cell holds the only constants.

In [ ]:
REPO = "yuosef772424/crypto-signal-prediction"
TOKEN_PATH = "/content/drive/MyDrive/crypto/secrets/github_token.txt"
QUEUE_BRANCH = "colab-queue"
RESULTS_BRANCH = "colab-results"
WORKDIR = "/content/csp_runner"

## 1. Mount Drive, get the token, fetch the job and the runner code
The token goes into an HTTP header only (never a URL, never printed). A missing, empty or rejected (401) token is asked for with
`getpass` and saved only after one authenticated GET to the repo accepts it. The runner module is downloaded from the exact commit the job pins.

In [ ]:
import getpass, json, os, re, sys, urllib.error, urllib.parse, urllib.request
from google.colab import drive

drive.mount("/content/drive")

def _api(url, tok, accept="application/vnd.github.raw"):
    req = urllib.request.Request(url, headers={"Authorization": "Bearer " + tok, "Accept": accept,
                                               "X-GitHub-Api-Version": "2022-11-28", "User-Agent": "colab-runner"})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read().decode("utf-8")

def _accepted(tok):
    """True if GitHub accepts the token for this repo (one GET); False on 401/404; any other failure stops the run."""
    try:
        _api(f"https://api.github.com/repos/{REPO}", tok, "application/vnd.github+json")
        return True
    except urllib.error.HTTPError as e:
        if e.code in (401, 404):
            return False
        raise RuntimeError(f"GitHub answered HTTP {e.code} while checking the token; try again later") from None
    except Exception as e:
        raise RuntimeError(f"cannot reach GitHub to check the token ({type(e).__name__})") from None

def _save(tok):
    os.makedirs(os.path.dirname(TOKEN_PATH), exist_ok=True)
    fd = os.open(TOKEN_PATH, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    with os.fdopen(fd, "w", encoding="utf-8") as fh:
        fh.write(tok + "\n")
    try:
        os.chmod(TOKEN_PATH, 0o600)
    except OSError:
        pass

try:
    _lines = [l.strip() for l in open(TOKEN_PATH, encoding="utf-8").read().splitlines() if l.strip()]
except OSError:
    _lines = []
_tok = _lines[0] if len(_lines) == 1 and len(_lines[0]) >= 8 and not any(c.isspace() for c in _lines[0]) else ""
if _tok and not _accepted(_tok):
    print(f"GitHub rejected the token saved in {TOKEN_PATH} (401: expired, revoked or no access to {REPO}). Enter a new one.")
    _tok = ""
elif not _tok:
    print(f"No usable token in {TOKEN_PATH}: paste one now (fine-grained, this repo only, Contents read/write). It is not shown.")
_new = ""
for _try in range(3 if not _tok else 0):
    _new = getpass.getpass("GitHub token: ").strip()
    if len(_new) >= 8 and not any(c.isspace() for c in _new) and _accepted(_new):
        _save(_new)
        _tok = _new
        print(f"Token accepted and saved to {TOKEN_PATH}; later runs read it silently.")
        break
    print("GitHub rejected that token (401 or no access to the repo). Try again.")
if not _tok:
    raise RuntimeError("no valid GitHub token after 3 attempts; see docs/research/plan/COLAB_RUNNER.md")

def _gh(path, ref=None, accept="application/vnd.github.raw"):
    url = f"https://api.github.com/repos/{REPO}/{path}" + (f"?ref={urllib.parse.quote(ref, safe='')}" if ref else "")
    try:
        return _api(url, _tok, accept)
    except Exception as e:
        raise RuntimeError(f"GitHub request for {path} failed: {type(e).__name__} {getattr(e, 'code', '')} "
                           "(token needs Contents read/write on this repo)") from None

_job_text = _gh("contents/job.json", QUEUE_BRANCH)
_queue_sha = _gh("commits/" + urllib.parse.quote(QUEUE_BRANCH, safe=""), accept="application/vnd.github.sha").strip()
try:
    _ref = str(json.loads(_job_text)["repo_ref"]).lower()
except Exception:
    raise RuntimeError("job.json is not valid JSON or has no repo_ref") from None
if not re.fullmatch(r"[0-9a-f]{40}", _ref):
    raise RuntimeError("job.json repo_ref must be a full 40-hex commit SHA")
os.makedirs(f"{WORKDIR}/bootstrap", exist_ok=True)
with open(f"{WORKDIR}/bootstrap/colab_runner.py", "w", encoding="utf-8") as fh:
    fh.write(_gh("contents/tools/colab_runner.py", _ref))
sys.path.insert(0, f"{WORKDIR}/bootstrap")
sys.modules.pop("colab_runner", None)
import colab_runner as R
del _tok, _new, _gh, _api, _accepted, _save, _lines
print("queue commit:", _queue_sha[:12], "| runner from:", _ref[:12])

## 2. Validate the job, refuse finished jobs, check out the pinned commit, install
A job id that already has `runs/<job_id>/` on `colab-results` is refused unless `job.json` sets `"rerun": true`.

In [ ]:
runner = R.Runner(R.github_url(REPO), R.read_token(TOKEN_PATH), WORKDIR, QUEUE_BRANCH, RESULTS_BRANCH)
runner.load(_job_text, _queue_sha)
runner.check_rerun()
runner.checkout()
runner.install()

## 3. Run the entry script (wall-clock guard, log streamed to a file)

In [ ]:
runner.execute()

## 4. Validate outputs, push to `colab-results`, copy to Drive, print the status
A failed run still pushes its log with status FAIL. The last block printed here is all that needs to be copied if anything is.

In [ ]:
result = runner.publish()